In [2]:
import numpy as np
import torch
torch.backends.cuda.matmul.allow_tf32 = False ## NOTE: important for consistency between cpu and cuda https://github.com/pytorch/pytorch/issues/77397
from NN import Net, NetBunny
from edge_subdivision import skeletal_subdivision
from domains import get_simplex, get_simplex_rightangle, get_hypercube

In [3]:
## TODO: double check stability, clean up
# del vs, v_sv, edges
### Define a NN ###
# torch.manual_seed(10); f = Net(ks=[784,48,48,1]); bbox = -1e-2, 1e-2
f = NetBunny(dim=3, depth=3, width=16)

### Domain 
D = f.ks[0]
domain = "simplex"
bbox = -1.6, 1.6 # bunny
# domain = "hypercube"
# bbox = -.5, .5 # bunny

if domain=="simplex_rightangle":
    ## SIMPLEX RIGHTANGLE
    center = torch.zeros(D) - 0.5
    vs, edges, v_sv = get_simplex_rightangle(center, bbox[1]-bbox[0])

elif domain=="simplex":
    # SIMPLEX EQUI
    center = torch.zeros(D)
    vs, edges, v_sv = get_simplex(center, bbox[1]-bbox[0])

elif domain=="hypercube":
    ## HYPERCUBE
    vs, edges, v_sv = get_hypercube(D, bbox)


## Store domain for plotting only
vs_init, v_sv_init, edges_init = vs, v_sv, edges
B = len(v_sv.T) # Number of hyperplanes


### Run subdivision ###
device = 'cuda'
vs = vs.to(device)
v_sv = v_sv.to(device)
edges = edges.to(device)

with torch.no_grad():
    # torch.rand(1, device='cuda') ## intialize cuda context, if you want to do automated timings
    # vs, edges, v_sv = skeletal_subdivision(f, device='cuda', plot=0, prune=0, bbox=bbox)
    vs, edges, v_sv = skeletal_subdivision(f, vs, v_sv, edges, device=device, plot=0, prune=0)


### Plot ###
# from utils_viz import plot_verts_and_edges
# from utils import get_labels
# e_sv = get_e_sv(v_sv, edges)
# plot_verts_and_edges(vs, edges, verts=1, bbox=bbox)
# plot_verts_and_edges(vs, edges, e_labels=get_labels(e_sv, B=B), verts=False, bbox=bbox)
# plot_verts_and_edges(vs, edges, e_labels=get_labels(e_sv, B=0), v_labels=get_labels(v_sv, B=0), bbox=bbox)
# plot_verts_and_edges(vs, edges, verts=0, edge_colors=np.where(e_sv[:,-1].cpu()==0, 'g', 'k'), bbox=bbox) ## Highlight the iso-edges

Layer 1: identified          749 vertices and        2,065 edges in total of    0.216s
Layer 2: identified        6,932 vertices and       20,061 edges in total of    0.236s
Layer 3: identified       29,808 vertices and       87,504 edges in total of    0.257s
Layer 4: identified       32,682 vertices and       96,126 edges in total of    0.258s


In [ ]:
## PLOT in 3D
import k3d

vs, edges, v_sv = vs.cpu(), edges.cpu(), v_sv.cpu()
e_sv = get_e_sv(v_sv, edges)

from complex import Complex
c = Complex(vs.detach(), edges.detach(), v_sv.detach(), e_sv.detach(), do_build_mesh_helpers=1)


fig = k3d.plot(height=1000, camera_fov=5, grid_visible=False)

## Formatting
color_strong = 0x0
color_ghost = 0x777777
color_highlight = 0x990099 # 0x7f00ff
lw = 0.07 ## line width
ps = .1 ## point size
op=.2

fig += k3d.lines(vs, edges, indices_type='segment', color=color_strong, width=lw, shader='simple', opacity=0.3)
fig += k3d.mesh(*c.get_all_face_mesh(-1), color=color_highlight, opacity=1)#opacity=1-((1-op)**2))
fig += k3d.lines(vs_init, edges_init, indices_type='segment', color=color_strong, width=lw*2, shader='simple', opacity=0.3)

fig.display()